# Homework — Part A

MILK10k: 5,240 skin lesions, each with one dermoscopic and one clinical close-up image.
Reusable code lives in `../src/` and is imported here; the same functions are used by the Part B scripts.
The data folder is read from the environment variable `MILK10K_DIR` (see README).

In [1]:
import sys
sys.path.append("..")

import io
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.data import load_metadata, load_gt, add_labels, image_path, filter_available

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
SEED = config.SEED
classes = config.CLASSES

meta = load_metadata()
gt = load_gt()
print("metadata:", meta.shape, " training_gt:", gt.shape)
meta.head(3)

metadata: (10480, 17)  training_gt: (5240, 12)


,isic_id,attribution,copyright_license,age_approx,anatom_site_general,anatom_site_special,concomitant_biopsy,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_4,diagnosis_confirm_type,image_manipulation,image_type,lesion_id,melanocytic,sex
0,ISIC_0051817,MILK study team,CC-BY-NC,70.0,upper extremity,NaN,True,Malignant,Malignant epidermal proliferations,"Squamous cell carcinoma, Invasive",NaN,histopathology,instrument only,dermoscopic,IL_1612768,NaN,male
1,ISIC_0073863,MILK study team,CC-BY-NC,5.0,NaN,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Reed",histopathology,instrument only,dermoscopic,IL_2547802,True,female
2,ISIC_0075884,MILK study team,CC-BY-NC,10.0,upper extremity,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Acral",histopathology,instrument only,clinical: close-up,IL_9270970,True,female


# A1 — pandas

## A1.1 Cross-check the two label files

In [2]:
# (a) same lesions in both files?
ids_meta, ids_gt = set(meta.lesion_id), set(gt.lesion_id)
print("only in metadata:", len(ids_meta - ids_gt), "| only in training_gt:", len(ids_gt - ids_meta))
assert ids_meta == ids_gt

only in metadata: 0 | only in training_gt: 0


In [3]:
# (b) exactly one class per row in the one-hot ground truth
row_sums = gt[classes].sum(axis=1)
print("rows that are not exactly one-hot:")
print(gt[row_sums != 1])
assert (row_sums == 1).all()

rows that are not exactly one-hot:
Empty DataFrame
Columns: [lesion_id, AKIEC, BCC, BEN_OTH, BKL, DF, INF, MAL_OTH, MEL, NV, SCCKA, VASC]
Index: []


In [4]:
# (c) 11-class label vs diagnosis_1, counted per lesion
labelled = add_labels(meta, gt)
lesions = labelled.drop_duplicates("lesion_id")
ct = pd.crosstab(lesions.dx, lesions.diagnosis_1)
print(ct)
multi = ct[(ct > 0).sum(axis=1) > 1]
print("\nclasses that map to more than one diagnosis_1 value:")
print(multi)

diagnosis_1  Benign  Indeterminate  Malignant
dx                                           
AKIEC             0            123        180
BCC               0              0       2522
BEN_OTH          44              0          0
BKL             544              0          0
DF               52              0          0
INF              50              0          0
MAL_OTH           0              0          9
MEL               0              0        450
NV              746              0          0
SCCKA             0              0        473
VASC             47              0          0

classes that map to more than one diagnosis_1 value:
diagnosis_1  Benign  Indeterminate  Malignant
dx                                           
AKIEC             0            123        180


In [5]:
# (d) the diagnosis hierarchy must be consistent: each diagnosis_3 -> one diagnosis_2, each diagnosis_2 -> one diagnosis_1
d3_to_d2 = meta.groupby("diagnosis_3").diagnosis_2.nunique()
d2_to_d1 = meta.groupby("diagnosis_2").diagnosis_1.nunique()
print("diagnosis_3 with >1 diagnosis_2:", (d3_to_d2 > 1).sum(), "| diagnosis_2 with >1 diagnosis_1:", (d2_to_d1 > 1).sum())
assert (d3_to_d2 == 1).all() and (d2_to_d1 == 1).all()
print("rows with missing diagnosis_3:", meta.diagnosis_3.isna().sum())

diagnosis_3 with >1 diagnosis_2: 0 | diagnosis_2 with >1 diagnosis_1: 0
rows with missing diagnosis_3: 158


In [6]:
# (e) lesion-level fields must be identical for the two images of a lesion
per_lesion = meta.groupby("lesion_id")[["age_approx", "sex", "anatom_site_general", "diagnosis_1"]].nunique(dropna=False)
print(per_lesion.max())
assert (per_lesion <= 1).all().all()

age_approx             1
sex                    1
anatom_site_general    1
diagnosis_1            1
dtype: int64


**Answer A1.1.** The two files describe exactly the same 5,240 lesions and every ground-truth row is a valid one-hot vector.
The diagnosis hierarchy is consistent (each `diagnosis_3` has one parent `diagnosis_2`, each `diagnosis_2` one `diagnosis_1`), and age, sex, site and `diagnosis_1` are identical for the two images of a lesion, so they are really lesion-level fields.
The one problem is (c): **AKIEC** maps to two `diagnosis_1` values (180 Malignant, 123 Indeterminate) — actinic keratosis / Bowen's disease is graded as either pre-malignant or malignant.
All other classes map to exactly one value. So `diagnosis_1` cannot always be derived from an 11-class prediction: a lesion predicted as AKIEC could be Malignant or Indeterminate, which is why we train on `diagnosis_1` directly (or keep both heads).

## A1.2 Five questions

In [7]:
# 1. BCC lesions in patients aged >= 70 on the head/neck
q1 = lesions[(lesions.dx == "BCC") & (lesions.age_approx >= 70) & (lesions.anatom_site_general == "head/neck")]
print("Q1: BCC, age >= 70, head/neck ->", q1.lesion_id.nunique(), "lesions")

Q1: BCC, age >= 70, head/neck -> 380 lesions


In [8]:
# 2. Which class has the highest share of 'altered' images?
altered = labelled.groupby("dx").image_manipulation.apply(lambda s: (s == "altered").mean())
print(f"Q2: highest share of altered images: {altered.idxmax()} ({altered.max():.1%}); overall {(labelled.image_manipulation == 'altered').mean():.1%}")

Q2: highest share of altered images: BEN_OTH (18.2%); overall 3.2%


In [9]:
# 3. Age range of melanoma patients
mel_age = lesions.loc[lesions.dx == "MEL", "age_approx"]
print(f"Q3: MEL age min {mel_age.min():.0f}, max {mel_age.max():.0f}, range {mel_age.max() - mel_age.min():.0f} years")

Q3: MEL age min 20, max 85, range 65 years


In [10]:
# 4. Does the class mix differ between lesions with and without a recorded site?
no_site = lesions.anatom_site_general.isna()
mix = pd.DataFrame({
    "site missing": lesions[no_site].dx.value_counts(normalize=True),
    "site known": lesions[~no_site].dx.value_counts(normalize=True),
}).fillna(0)
mix["diff_pp"] = (mix["site missing"] - mix["site known"]) * 100
print((mix * [100, 100, 1]).round(1).sort_values("diff_pp"))
biggest = mix.diff_pp.abs().idxmax()
print(f"Q4: largest difference: {biggest}, {mix.loc[biggest, 'diff_pp']:+.1f} percentage points ({no_site.mean():.0%} of lesions have no site)")

         site missing  site known  diff_pp
dx                                        
SCCKA             2.9        12.7     -9.8
AKIEC             2.0         8.0     -6.0
DF                0.5         1.3     -0.8
BKL              10.1        10.6     -0.5
INF               0.9         1.0     -0.1
BEN_OTH           0.8         0.9     -0.1
MAL_OTH           0.3         0.1      0.2
VASC              1.2         0.7      0.4
BCC              49.3        47.4      1.9
MEL              11.0         7.2      3.8
NV               21.1        10.1     11.0
Q4: largest difference: NV, +11.0 percentage points (37% of lesions have no site)


In [11]:
# 5. Confirmation type vs diagnosis_1
print((pd.crosstab(lesions.diagnosis_confirm_type, lesions.diagnosis_1, normalize="index") * 100).round(1))

diagnosis_1                             Benign  Indeterminate  Malignant
diagnosis_confirm_type                                                  
histopathology                            25.7            2.0       72.3
single contributor clinical assessment    87.1           10.7        2.2


**Answer Q5.** Histopathology-confirmed (biopsied) lesions are mostly Malignant, while lesions confirmed only by a clinician's assessment are almost all Benign.
A lesion is biopsied because it looked suspicious, so a dataset built mainly from biopsied lesions is **enriched with malignant cases** (selection bias): the class mix is far more malignant than what a GP sees in daily practice.